In [11]:
import srfm
import srfm.main
import inspect
import importlib.metadata
import numpy as np
import matplotlib.pyplot as plt
"""If running the SRFM with a driver table, run this code."""
from srfm import *
import xarray as xr
from scipy.interpolate import interp1d
import os
from scipy.signal import convolve
import pandas as pd
import json
import csv
from prettytable import PrettyTable as pt

# 1. Selecting Ash properties

In [14]:
"""
This cell identifies and saves SRFM spectra whose ash properties match
the selected target values.

SRFM Zarr datasets are searched across all profiles stored in
`base_path` and for both spectral bands (`a` and `b`). For each spectrum,
the `srfm_params` metadata are read and the ash properties are extracted
from the `Ash_1` scattering-layer input.

The matching spectra are written to a CSV file containing the spectrum
index and the corresponding ash properties.

Input: select ash properties eg:
--------------------------
mass_loading_values : list of float
    Target ash mass loadings:
        [0.5, 7.0, 20.0, 50.0]

alt_upp_values : list of float
    Target cloud top altitudes:
        [10.714, 7.7970]

r_value : float
    Target ash median number radius:
        0.62

Output:
------
For each Zarr profile and spectral band(group a and b; note group a and b are identical), a csv file is created with
the following columns:

    index
        [Index of the matching spectrum in the Zarr dataset.]
    mass_loading
    r
    alt_low

    alt_upp

The output files are saved in:
    /home/m/moulin-garrigues/SRFM_RTTOV/Mes_resultats/Matching_fr_fr_alt

Matching criterion
------------------
A spectrum is selected when:

    mass_loading ∈ mass_loading_values
    alt_upp ∈ alt_upp_values
    r ≈ r_value

Workflow
--------
1. Loop through Zarr profiles files in `base_path`
2. Open both spectral groups (`a` and `b`)
3. Load `srfm_params` into memory
4. Loop through every spectrum in the dataset
5. Extract the `Ash_1` properties from the spectrum metadata
6. Compare properties with the selected target values
7. Store the indices of matching spectra
8. Write the matching indices and their properties into a csv file

These csv files will be used to compute radiances for indices matching the selected properties.
"""
base_path = "/network/group/aopp/eodg/RGG008_GRAINGER_IASIVOLC/knizek/MetOffice_intercomparison_dataset/zarr"

output_dir = "/home/m/moulin-garrigues/SRFM_RTTOV/Mes_resultats/Matching_fr_fr_alt"
os.makedirs(output_dir, exist_ok=True)

#Select your ash properties 
mass_loading_values = [0.5, 7.0, 20.0, 50.0] 
alt_upp_values = [10.714, 7.7970]
r_value = 0.62

for zarr_name in sorted(os.listdir(base_path)):

    zarr_path = os.path.join(base_path, zarr_name)

    print("\n==============================")
    print(f"Processing {zarr_name}")
    print("==============================")

    for group in ["a", "b"]:

        ds = xr.open_zarr(
            zarr_path,
            group=group,
            consolidated=False
        )

        # Load srfm_params into memory so .item() works
        srfm_params = ds["srfm_params"].load()

        #Name of outputed csv file with ash properties
        output_file = os.path.join(
            output_dir,
            f"matches_{zarr_name.replace('.zarr','')}_{group}_9.3.csv"
        )

        matches = []

        with open(output_file, "w", newline="") as f:

            writer = csv.writer(f)

            writer.writerow([
                "index",
                "mass_loading",
                "r",
                "alt_low",
                "alt_upp"
            ])

            print(f"\nGroup: {group}")

            for i in range(ds.sizes["spectrum"]):

                params = json.loads(
                    srfm_params.isel(spectrum=i).item()
                )

                try:
                    ash = params["scat_lyrs_inputs"]["Ash_1"]

                    mass_loading = ash["mass_loading"]
                    alt_upp = ash["alt_upp"]
                    r = ash["r"]
                    alt_low = ash["alt_low"]

                    if (
                        np.isclose(mass_loading, mass_loading_values).any()
                        and np.isclose(alt_upp, alt_upp_values).any()
                        and np.isclose(r, r_value)
                    ):

                        matches.append(i)

                        writer.writerow([
                            i,
                            mass_loading,
                            r,
                            alt_low,
                            alt_upp
                        ])

                        print(
                            f"spectrum index={i}: "
                            f"mass_loading={mass_loading}, "
                            f"alt_upp={alt_upp}, "
                            f"r={r}"
                        )

                except KeyError:
                    continue

        print(f"Found {len(matches)} matching spectra")
        print(f"Saved: {output_file}")



Processing 0.zarr

Group: a
spectrum index=1050: mass_loading=0.5, alt_upp=7.797, r=0.62
spectrum index=1051: mass_loading=0.5, alt_upp=7.797, r=0.62
spectrum index=1052: mass_loading=0.5, alt_upp=7.797, r=0.62
spectrum index=1053: mass_loading=0.5, alt_upp=7.797, r=0.62
spectrum index=1054: mass_loading=0.5, alt_upp=7.797, r=0.62
spectrum index=1060: mass_loading=0.5, alt_upp=10.714, r=0.62
spectrum index=1061: mass_loading=0.5, alt_upp=10.714, r=0.62
spectrum index=1062: mass_loading=0.5, alt_upp=10.714, r=0.62
spectrum index=1063: mass_loading=0.5, alt_upp=10.714, r=0.62
spectrum index=1064: mass_loading=0.5, alt_upp=10.714, r=0.62
spectrum index=4620: mass_loading=7, alt_upp=7.797, r=0.62
spectrum index=4621: mass_loading=7, alt_upp=7.797, r=0.62
spectrum index=4622: mass_loading=7, alt_upp=7.797, r=0.62
spectrum index=4623: mass_loading=7, alt_upp=7.797, r=0.62
spectrum index=4624: mass_loading=7, alt_upp=7.797, r=0.62
spectrum index=4630: mass_loading=7, alt_upp=10.714, r=0.62
s

# 2. Computing weighted channel averaged radiances for the spectra corresponding to the selected ash properties

## a.

In [2]:
#This cell loads the spectral response functions for MSG_3 SEVIRI

#Load Data - Channels_3
srf_file_7_ = np.loadtxt(
    "/home/m/moulin-garrigues/rtcoef_msg_3_seviri_srf_ch07.txt",
    skiprows=4)
#Channel file9
srf_file_9_= np.loadtxt(
    "/home/m/moulin-garrigues/rtcoef_msg_3_seviri_srf_ch09.txt",
    skiprows=4)
#Channel file10
srf_file_10_ = np.loadtxt(
    "/home/m/moulin-garrigues/rtcoef_msg_3_seviri_srf_ch10.txt",
    skiprows=4)
#Channel file11
srf_file_11_= np.loadtxt(
    "/home/m/moulin-garrigues/rtcoef_msg_3_seviri_srf_ch11.txt",
    skiprows=4)


In [9]:
#This function computes channel averaged radiances weighted by the spectral response functions for each channel
def channel_radiance_multi(spec_a, spec_b, srf_files_a, srf_files_b):
    """
    The function supports spectra from two spectral bands, `spec_a` and
    `spec_b`. Each SRF is interpolated onto the wavenumber grid of the
    corresponding spectrum. The channel radiance is then calculated as
    the SRF-weighted average of the spectral radiance:

        R_channel = ∫ R(ν) SRF(ν) dν / ∫ SRF(ν) dν

    where:
        R(ν)   is SRFM spectral radiance,
        SRF(ν) is spectral response function,
        ν      is wavenumber.

    Input Parameters
    ----------
    spec_a : xarray.DataArray
        Radiance for spectral band a. The DataArray must contain
        a `wavenumber` coordinate.

    spec_b : xarray.DataArray
        Radiance for spectral band b. The DataArray must contain
        a `wavenumber` coordinate.

    srf_files_a : list of numpy.ndarray
        List of SRFs for channels in spectral band a (e.g. channel 7). Each SRF is expected
        to be a two-column array:
            column 0: wavenumber
            column 1: spectral response weight

    srf_files_b : list of numpy.ndarray
        List of SRFs for channels in spectral band b (e.g. channel 9,10,11). Each SRF is expected
        to be a two-column array:
            column 0: wavenumber
            column 1: spectral response weight
    Returns
    -------
    radiances : list of float
        Weighted channel averaged radiances. The values are returned in the same
        order as the SRFs supplied in `srf_files_a` followed by those
        supplied in `srf_files_b`.

        e.g. for my application:
            [Channel 07, Channel 09, Channel 10, Channel 11]

    plots : list of numpy.ndarray
        SRF-weighted spectral radiances for each channel, calculated as:

            spectral_radiance * interpolated_SRF
            This is to plot the "convolved" spectrum.
    -----
    The SRF is linearly interpolated onto the wavenumber grid
    using `scipy.interpolate.interp1d`. Wavenumbers outside the SRF
    range are assigned a response of zero.
"""

    def integrate_channel_(spec, srf_file):
    
        wv = spec.wavenumber.values #wavenumber values of SRFM spectra (note these are stored in the SRFM file, unlike for clear sky)
        spc = spec.values #wavenumber values of SRFM spectra
    
        ils_x = srf_file[:, 0]
        ils_y = srf_file[:, 1]
    
        srf = interp1d(
            ils_x,
            ils_y,
            bounds_error=False,
            fill_value=0.0
        )(wv)
    
        plot = spc * srf
        
        rad = np.trapezoid(plot, wv) / np.trapezoid(srf, wv)
    
        return rad, plot
    
    radiances = []
    plots = []
    
    for srf_file in srf_files_a:
        rad, plot = integrate_channel_(spec_a, srf_file)
        radiances.append(rad)
        plots.append(plot)
    
    for srf_file in srf_files_b:
        rad, plot = integrate_channel_(spec_b, srf_file)
        radiances.append(rad)
        plots.append(plot)
    
    rad_ch07, rad_ch09, rad_ch10, rad_ch11 = radiances
    print(radiances)
    
    return radiances, plots

## b. The cell below takes a profile eg.pro=8; loads the indices of the spectra corresponding to the selected ash properties; computes weighted channel averaged radiances for this profile and these indices (ie.your selected ash properties); and inputs these radiances into a csv file with headers for each channel

In [6]:
"""This cell computes channel averaged radiances using channel_radiance_multi which have matching indices with the properties we are interested in;
and inputs these values into a CSV file. The csv file generated in "1.Selecting ash properties" is used to select the indices corresponding to these properties and compute radiance for these indices.
Note the warnings 'Failed to open zarr files with consolidated metadata...' are normal.
The cell takes a while to run e.g. about 40mins for 40 matching indices for a given profile (ie computing 40 x4 [channels] radiances).


Input:
    - Zarr datasets for given profile containing SRFM spectra for bands
      `a` and `b`.
    - csv file containing the indices of spectra matching the selected
      ash properties (see part 1.).
    - Spectral response functions for channels 07, 09, 10 and 11.

Output:
    A CSV file containing the spectrum index and channel-averaged
    radiances for 4 channels.
    
Processing:
    Matching spectra are loaded in blocks to limit memory usage.
    Each spectrum is processed individually and its four channel
    radiances are calculated using `channel_radiance_multi()`.
"""
pro = 8 #Select profile number e.g. profile 8
path = f"/network/group/aopp/eodg/RGG008_GRAINGER_IASIVOLC/knizek/MetOffice_intercomparison_dataset/zarr/{pro}.zarr" 
ds_a = xr.open_zarr(path, group="a") 
ds_b = xr.open_zarr(path, group="b") #These lines open the zarr files for the two srfm spectral bands a and b

srf_files_a = [srf_file_7_] # this is the srf file which belongs in the srfm band a
srf_files_b = [srf_file_9_, srf_file_10_, srf_file_11_] # these are the srf file which belongs in the srfm band b

# Load matching indices from CSV file containing the properties we are interested in (eg. 4 mass loadings, 2 alt) - generated in part 1.
matches = pd.read_csv(
    f"/home/m/moulin-garrigues/SRFM_RTTOV/Mes_resultats/Matching_fr_fr_alt/matches_{pro}_a_9.3.csv"
)

matching_indices = matches["index"].tolist()

block_size = 50
width = 27

output_file = f'/home/m/moulin-garrigues/SRFM_RTTOV/Mes_resultats/Combined_prop_rad_alt/Matches_{pro}_with_radiances_alt.csv' #name of output file

with open(output_file, "w") as f:

    f.write(
        f"{'ID':<15}"
        f"{'Channel 11 (13.4 µm)':>{width}}"
        f"{'Channel 10 (12 µm)':>{width}}"
        f"{'Channel 09 (10.8 µm)':>{width}}"
        f"{'Channel 07 (8.7 µm)':>{width}}\n"
    )

    f.write("-" * (15 + 4*width) + "\n")


    # Loop only over matching indices
    for start in range(0, len(matching_indices), block_size):

        block_indices = matching_indices[start:start + block_size]

        print(f"Processing indices {block_indices}")

        # Load only matching spectra from zarr
        block_a = ds_a["rad"].isel(
            spectrum=block_indices
        )

        block_b = ds_b["rad"].isel(
            spectrum=block_indices
        )


        # Process spectra one by one
        for j, i in enumerate(block_indices):

            rad_a = block_a.isel(spectrum=j)
            rad_b = block_b.isel(spectrum=j)


            radiances, _ = channel_radiance_multi(
                rad_a,
                rad_b,
                srf_files_a,
                srf_files_b)


            # radiances order:
            # [Channel 07, Channel 09, Channel 10, Channel 11]
            rad_ch07, rad_ch09, rad_ch10, rad_ch11 = radiances


            # Write result
            f.write(
                f"{i:<15}"
                f"{rad_ch11:>{width}.6f}"
                f"{rad_ch10:>{width}.6f}"
                f"{rad_ch09:>{width}.6f}"
                f"{rad_ch07:>{width}.6f}\n"
            )

/tmp/ipykernel_446028/2569620792.py:3: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  ds_a = xr.open_zarr(path, group="a")
/tmp/ipykernel_446028/2569620792.py:4: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to rea

Processing indices [1050, 1051, 1052, 1053, 1054, 1060, 1061, 1062, 1063, 1064, 4620, 4621, 4622, 4623, 4624, 4630, 4631, 4632, 4633, 4634, 8190, 8191, 8192, 8193, 8194, 8200, 8201, 8202, 8203, 8204, 10230, 10231, 10232, 10233, 10234, 10240, 10241, 10242, 10243, 10244]
[np.float64(0.046383998939872974), np.float64(0.08184611121528669), np.float64(0.10087967213753829), np.float64(0.07425672560472094)]
[np.float64(0.04668936552661587), np.float64(0.08240466764069301), np.float64(0.10112624690875335), np.float64(0.07433025917957407)]
[np.float64(0.04705017643018667), np.float64(0.08314938432441946), np.float64(0.10137265033227323), np.float64(0.07421618998519999)]
[np.float64(0.04759436039906216), np.float64(0.08413802239634316), np.float64(0.10172292800910501), np.float64(0.07408246813421171)]
[np.float64(0.04788124284705474), np.float64(0.08459616148431015), np.float64(0.10188801366209023), np.float64(0.07353491379175003)]


KeyboardInterrupt: 

### The cell below is the same as above, only looping over all the profiles 

In [7]:
"""This cell is the same as above, only it runs the matches code looping over all profiles (here 10profiles)"""

for pro in range(10):

    print("\n" + "=" * 60)
    print(f"Starting profile {pro}")
    print("=" * 60)

    path = (
        f"/network/group/aopp/eodg/RGG008_GRAINGER_IASIVOLC/"
        f"knizek/MetOffice_intercomparison_dataset/zarr/{pro}.zarr"
    )

    spec_name = f"{pro}.zarr"

    ds_a = xr.open_zarr(path, group="a")
    ds_b = xr.open_zarr(path, group="b")

    srf_files_a = [srf_file_7_]
    srf_files_b = [srf_file_9_, srf_file_10_, srf_file_11_]

    # Load matching indices from properties CSV file
    matches = pd.read_csv(
        f"/home/m/moulin-garrigues/SRFM_RTTOV/Mes_resultats/"
        f"Matching_fr_fr_alt/matches_{pro}_a_9.3.csv"
    )

    matching_indices = matches["index"].tolist()

    block_size = 50
    width = 27

    output_file = (
        f"/home/m/moulin-garrigues/SRFM_RTTOV/Mes_resultats/"
        f"Combined_prop_rad_alt/retry_matches_{pro}_with_radiances_alt.csv"
    )

    with open(output_file, "w") as f:

        f.write(
            f"{'ID':<15}"
            f"{'Channel 11 (13.4 µm)':>{width}}"
            f"{'Channel 10 (12 µm)':>{width}}"
            f"{'Channel 09 (10.8 µm)':>{width}}"
            f"{'Channel 07 (8.7 µm)':>{width}}\n"
        )

        f.write("-" * (15 + 4 * width) + "\n")

        # Loop over matching indices in blocks
        for start in range(0, len(matching_indices), block_size):

            block_indices = matching_indices[start:start + block_size]

            print(
                f"Profile {pro}: processing indices "
                f"{start + 1}-{min(start + block_size, len(matching_indices))} "
                f"of {len(matching_indices)}"
            )

            block_a = ds_a["rad"].isel(
                spectrum=block_indices
            )

            block_b = ds_b["rad"].isel(
                spectrum=block_indices
            )

            # Process spectra one by one
            for j, i in enumerate(block_indices):

                rad_a = block_a.isel(spectrum=j)
                rad_b = block_b.isel(spectrum=j)

                radiances, _ = channel_radiance_multi(
                    rad_a,
                    rad_b,
                    srf_files_a,
                    srf_files_b)

                # [Channel 07, Channel 09, Channel 10, Channel 11]
                rad_ch07, rad_ch09, rad_ch10, rad_ch11 = radiances

                f.write(
                    f"{i:<15}"
                    f"{rad_ch11:>{width}.6f}"
                    f"{rad_ch10:>{width}.6f}"
                    f"{rad_ch09:>{width}.6f}"
                    f"{rad_ch07:>{width}.6f}\n"
                )

    # Close datasets before moving to next profile
    ds_a.close()
    ds_b.close()

    print(f"Finished profile {pro}")
    print(f"Output saved to: {output_file}")

print("\n" + "=" * 60)
print("ALL 10 PROFILES FINISHED")
print("=" * 60)



Starting profile 0


/tmp/ipykernel_3876903/311222429.py:14: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  ds_a = xr.open_zarr(path, group="a")
/tmp/ipykernel_3876903/311222429.py:15: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to r

Profile 0: processing indices 1-40 of 40


KeyboardInterrupt: 

# 3. Merge file containing properties (1.) with file containing equivalent channel radiances (2.) - ie create a csv file with both

In [15]:
matching_dir = "/home/m/moulin-garrigues/SRFM_RTTOV/Mes_resultats/Matching_fr_fr_9.3" #file created in part 1. with the selected properties
radiance_dir = "/home/m/moulin-garrigues/SRFM_RTTOV/Mes_resultats/Combined_prop_rad_9.3" #file created in part 2. with the corresponding radiances

for pro in range(10):  # profiles 0–9

    print(f"\nProcessing profile {pro}")

    # Read properties
    properties = pd.read_csv(
        f"{matching_dir}/matches_{pro}_a_9.3.csv"
    )

    # Read radiances
    radiances = pd.read_csv(
        f"{radiance_dir}/retry_matches_{pro}_with_radiances_9.3.csv",
        sep=r"\s+",
        skiprows=2,
        header=None,
        names=["index", "ch11", "ch10", "ch09", "ch07"],
        engine="python"
    )

    # Merge properties and radiances using the spectrum index
    combined = properties.merge(
        radiances,
        on="index",
        how="inner"
    )

    # Save final file - Give a name to output file
    output_file = (
        f"{radiance_dir}/matches_{pro}_with_radiances_9.3.csv"
    )

    combined.to_csv(
        output_file,
        index=False
    )

    print(f"Number of matched spectra: {len(combined)}")
    print(f"Saved: {output_file}")



Processing profile 0
Number of matched spectra: 40
Saved: /home/m/moulin-garrigues/SRFM_RTTOV/Mes_resultats/Combined_prop_rad_9.3/matches_0_with_radiances_9.3.csv

Processing profile 1
Number of matched spectra: 40
Saved: /home/m/moulin-garrigues/SRFM_RTTOV/Mes_resultats/Combined_prop_rad_9.3/matches_1_with_radiances_9.3.csv

Processing profile 2
Number of matched spectra: 40
Saved: /home/m/moulin-garrigues/SRFM_RTTOV/Mes_resultats/Combined_prop_rad_9.3/matches_2_with_radiances_9.3.csv

Processing profile 3


FileNotFoundError: [Errno 2] No such file or directory: '/home/m/moulin-garrigues/SRFM_RTTOV/Mes_resultats/Combined_prop_rad_9.3/retry_matches_3_with_radiances_9.3.csv'

# 4.